# Bab 16: Klasterisasi Hierarkis dan Self-Organizing Map

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab16/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab16")
print("Folder kerja:", os.getcwd())

## Listing 16.1: Single dan complete linkage pada enam gerai

`kode/bab16/linkage_mini.py`

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import pdist, squareform

nama = ["G1", "G2", "G3", "G4", "G5", "G6"]
X = np.array([[0.85, 0.80], [0.90, 0.85], [0.85, 1.00],
              [1.15, 1.00], [1.10, 1.05], [1.20, 0.95]])
D = squareform(pdist(X))
print("matriks jarak:")
print("     " + "".join(f"{n:>6s}" for n in nama))
for n, r in zip(nama, D):
    print(f"{n:4s} " + "".join(f"{v:6.3f}" for v in r))
for metode in ["single", "complete"]:
    Z = linkage(X, metode)
    print(f"\n{metode} linkage:")
    kl = {i: [nama[i]] for i in range(6)}
    for t, (a, b, h, n) in enumerate(Z):
        kl[6 + t] = kl[int(a)] + kl[int(b)]
        print(f"  langkah {t + 1}: gabung {{{','.join(kl[int(a)])}}} + "
              f"{{{','.join(kl[int(b)])}}} pada jarak {h:.3f}")

## Listing 16.2: Inversi pada linkage centroid

`kode/bab16/nonmonoton.py`

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage

X = np.array([[0.0, 0.0], [2.0, 0.0], [1.0, 1.8]])
for metode in ["centroid", "single", "complete", "average", "ward"]:
    Z = linkage(X, metode)
    print(f"{metode:9s} tinggi gabungan: {Z[0, 2]:.3f} lalu {Z[1, 2]:.3f}"
          + ("   <- turun (inversi)" if Z[1, 2] < Z[0, 2] else ""))

## Listing 16.3: SOM satu dimensi dengan tangan

`kode/bab16/som_mini.py`

In [ ]:
import numpy as np

W = np.array([[0.2, 0.8], [0.5, 0.5], [0.8, 0.2]])
X = np.array([[0.1, 0.9], [0.9, 0.1], [0.2, 0.7], [0.8, 0.3]])
eta, r = 0.5, 1
for t, x in enumerate(X[:2], 1):
    d = np.linalg.norm(W - x, axis=1)
    j = d.argmin()
    print(f"masukan {t} x={x}: jarak {d.round(3)}, pemenang neuron {j + 1}")
    for i in range(3):
        if abs(i - j) <= r:
            h = 1.0 if i == j else 0.5
            W[i] += eta * h * (x - W[i])
    print("  bobot baru:", "  ".join(f"({a:.3f}; {b:.3f})" for a, b in W))


def latih(X, W, n=200, eta0=0.5, r0=1.0, seed=0):
    rng = np.random.default_rng(seed)
    for t in range(n):
        x = X[rng.integers(len(X))]
        eta = eta0 * (1 - t / n)
        r = r0 * (1 - t / n)
        j = np.linalg.norm(W - x, axis=1).argmin()
        for i in range(len(W)):
            h = np.exp(-((i - j) ** 2) / (2 * max(r, 0.1) ** 2))
            W[i] += eta * h * (x - W[i])
    return W


print("setelah 200 langkah:",
      "  ".join(f"({a:.2f}; {b:.2f})" for a, b in latih(X, W)))

## Listing 16.4: SOM $8times8$ untuk anggota

`kode/bab16/som.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab16')
import sys
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
sys.path.insert(0, "kode/bab15")
from siapkan import PERILAKU, muat

df, X = muat()
Z = StandardScaler().fit_transform(X)
baris, kolom = 8, 8
rng = np.random.default_rng(0)
W = rng.normal(0, 0.1, (baris * kolom, Z.shape[1]))
pos = np.array([(i, j) for i in range(baris) for j in range(kolom)])
T = 20 * len(Z)
for t in range(T):
    x = Z[rng.integers(len(Z))]
    eta = 0.5 * (0.01 / 0.5) ** (t / T)
    sigma = 4.0 * (0.5 / 4.0) ** (t / T)
    j = ((W - x) ** 2).sum(1).argmin()
    h = np.exp(-((pos - pos[j]) ** 2).sum(1) / (2 * sigma ** 2))
    W += eta * h[:, None] * (x - W)
bmu = ((Z[:, None, :] - W[None]) ** 2).sum(2).argmin(1)
qe = np.sqrt(((Z - W[bmu]) ** 2).sum(1)).mean()
print(f"galat kuantisasi rata-rata {qe:.3f}; neuron kosong "
      f"{(np.bincount(bmu, minlength=64) == 0).sum()}")
km = KMeans(5, n_init=20, random_state=0).fit(Z)
seg = {0: "mahasiswa", 1: "pekerja", 2: "keluarga", 3: "tidur",
       4: "penikmat"}
mayor = pd.Series(km.labels_).groupby(bmu).agg(lambda s: s.mode()[0])
umat = np.zeros(baris * kolom)
for i, (a, b) in enumerate(pos):
    tetangga = [k for k, (c, e) in enumerate(pos)
                if abs(a - c) + abs(b - e) == 1]
    umat[i] = np.mean([np.linalg.norm(W[i] - W[k]) for k in tetangga])
with open("gambar/bab16/som.dat", "w") as f:
    f.write("i j n seg u\n")
    for i, (a, b) in enumerate(pos):
        f.write(f"{b} {a} {(bmu == i).sum()} {mayor.get(i, -1)} "
                f"{umat[i]:.3f}\n")
print("segmen dominan per baris peta (0-4, - = kosong):")
for a in range(baris):
    print("  " + " ".join(str(mayor.get(a * kolom + b, "-"))
                          for b in range(kolom)))
w = pd.DataFrame(W, columns=PERILAKU)
print("neuron dengan porsi kopi hitam tertinggi:",
      int(w["porsi_kopi_hitam"].idxmax()), "; terendah frekuensi:",
      int(w["frekuensi_90h"].idxmin()))

## Listing 16.5: Studi kasus klasterisasi gerai

`kode/bab16/studi_kasus.py`

In [ ]:
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import cophenet, fcluster, linkage
from scipy.spatial.distance import pdist
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler
from pola_gerai import indeks_gerai

F, gerai = indeks_gerai()
print(F.groupby(gerai["tipe_lokasi"]).mean().round(3).to_string())
lama = gerai.index[gerai["tanggal_buka"] < "2025-01-01"]
POLA = ["libur_nasional", "libur_semester", "ramadan"]
X = F.loc[lama, POLA]
sk = StandardScaler().fit(X)
Z = sk.transform(X)
d = pdist(Z)
print(f"\n{len(lama)} gerai dibuka sebelum 2025; korelasi kofenetik:")
for m in ["single", "complete", "average", "ward"]:
    print(f"  {m:8s} {cophenet(linkage(Z, m), d)[0]:.3f}")
L = linkage(Z, "ward")
print("tinggi 6 penggabungan terakhir:", np.round(L[-6:, 2], 2))
for k in range(2, 7):
    lab = fcluster(L, k, "maxclust")
    print(f"k={k}: silhouette {silhouette_score(Z, lab):.3f}, ARI vs "
          f"tipe lokasi {adjusted_rand_score(gerai.loc[lama, 'tipe_lokasi'], lab):.3f}")
kl = pd.Series(fcluster(L, 4, "maxclust"), index=lama, name="klaster")
print("\n" + pd.crosstab(kl, gerai.loc[lama, "tipe_lokasi"]).to_string())
print("\nprofil klaster (indeks, 1 = sama dengan hari biasa):")
print(F.loc[lama, POLA].groupby(kl).mean().round(3).to_string())
for g in lama:
    mayor = gerai.loc[kl.index[kl == kl[g]], "tipe_lokasi"].mode()[0]
    if gerai.loc[g, "tipe_lokasi"] != mayor:
        print(f"menyimpang: {g} {gerai.loc[g, 'nama_gerai']} "
              f"({gerai.loc[g, 'tipe_lokasi']}) di klaster {kl[g]}")
pusat = pd.DataFrame(Z, index=lama).groupby(kl).mean()
for g in gerai.index.difference(lama):
    z = sk.transform(F.loc[[g], POLA])
    k = ((pusat.to_numpy() - z) ** 2).sum(1).argmin() + 1
    print(f"gerai baru {g} ({gerai.loc[g, 'tipe_lokasi']}, buka "
          f"{gerai.loc[g, 'tanggal_buka']}): terdekat ke klaster {k}")
np.save("kode/bab16/linkage_gerai.npy", L)
kl.to_csv("kode/bab16/klaster_gerai.csv")